In [ ]:
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import pickle
import math
import igraph as ig
import seaborn as sns
import statistics
from scipy.sparse import csr_matrix, load_npz
from scipy.stats import rankdata
from matplotlib.patches import Patch
import matplotlib.cm as cm
from scipy.stats import mannwhitneyu

# Construct graphs

In [ ]:
def igraph_from_sparse(M, directed=False, weighted=True):
    """
    Build an igraph Graph from a SciPy sparse matrix.

    Parameters
    ----------
    M : scipy.sparse matrix (CSR/CSC/COO)
        Adjacency or weighted adjacency matrix.
    directed : bool
        Whether the graph is directed.
    weighted : bool
        Whether to store values as edge weights.

    Returns
    -------
    ig.Graph
    """
    M = M.tocoo()

    edges = list(zip(M.row.tolist(), M.col.tolist()))
    g = ig.Graph(edges=edges, directed=directed)

    if weighted:
        g.es["weight"] = M.data.tolist()

    return g

# Community preprocessing
def nx_to_igraph(G: nx.Graph, weight: str | None = "weight") -> ig.Graph:
    """
    Convert a NetworkX graph to an iGraph graph.
    
    Parameters
    ----------
    G : nx.Graph or nx.DiGraph
        Your NetworkX graph.
    weight : str or None, optional
        Name of the edge attribute to treat as weight.
        If None, graph is treated as unweighted.

    Returns
    -------
    ig.Graph
        An iGraph object with:
        - g.vs['name'] = node labels
        - g.es['weight'] = weights (if provided)
    """
    # 1) Keep original node labels
    nodes = list(G.nodes())
    idx_map = {node: i for i, node in enumerate(nodes)}

    # 2) Convert edges
    edges = [(idx_map[u], idx_map[v]) for u, v in G.edges()]

    # 3) Initialize iGraph
    g_ig = ig.Graph(edges=edges, directed=G.is_directed())
    g_ig.vs["name"] = nodes

    # 4) Add weights if available
    if weight is not None:
        # Extract weights or default to 1.0
        weights = [G[u][v].get(weight) for u, v in G.edges()]
        g_ig.es["weight"] = weights

    return g_ig

def intramodule_closeness(G, community_nodes, weight="weight"):
    """
    Intramodule closeness centrality using the induced subgraph of a community.
    
    Parameters
    ----------
    G : nx.Graph
        Original graph.
    community : iterable
        Nodes in the community (subset of G).
    weight : str or None
        Edge attribute to use as weights.
    
    Returns
    -------
    dict
        Mapping node -> intramodule closeness centrality.
    """
    # Induced subgraph
    sub = G.subgraph(community_nodes)

    # Regular closeness, but only inside H
    cl = sub.closeness(weights=weight, normalized=True)
    closeness = {sub.vs[i]["name"]: float(cl[i]) for i in range(sub.vcount())}
    return closeness

def degree_for_community(G, community_nodes, weight="weight"):
    """
    Intramodule closeness centrality using the induced subgraph of a community.
    
    Parameters
    ----------
    G : nx.Graph
        Original graph.
    community : iterable
        Nodes in the community (subset of G).
    weight : str or None
        Edge attribute to use as weights.
    
    Returns
    -------
    dict
        Mapping node -> intramodule closeness centrality.
    """
    # Induced subgraph
    sub = G.subgraph(community_nodes)

    # Regular closeness, but only inside H
    cl = sub.strength(weights=weight)
    strength = {sub.vs[i]["name"]: float(cl[i]) for i in range(sub.vcount())}
    return strength

def pagerank_for_community(G, community_nodes, weight="weight"):
    # 1. Build subgraph by vertex names
    sub = G.subgraph(community_nodes).copy()

    # 2. Compute PageRank
    pr = nx.pagerank(sub, weight=weight)

    return pr

def eig_centrality_for_community(G, community_nodes, weight="weight"):
    # 1. Build subgraph by vertex names
    sub = G.subgraph(community_nodes).copy()

    # 2. Compute PageRank
    pr = nx.eigenvector_centrality(sub, weight=weight, max_iter=5000, tol=1e-6)

    return pr

def betweenness_for_community(
    g: ig.Graph,
    community_nodes,
    weight: str | None = "weight"
):
    """
    Compute betweenness centrality for nodes inside a community.

    Parameters
    ----------
    g : ig.Graph
        Full igraph graph (must have vs['name']).
    community_nodes : list of str
        Node names belonging to this community.
    weight : str or None
        Edge weight attribute name. None = unweighted betweenness.
    normalized : bool
        Normalize betweenness by maximum possible value.

    Returns
    -------
    dict {node_name : betweenness_score}
    """

    # Build subgraph of the community
    sub = g.subgraph(community_nodes)
    sub.vs["orig_vid"] = community_nodes
    n = sub.vcount()

    w = np.asarray(sub.es[weight], dtype=float)

    # Rescale so the smallest weight becomes 1.0 (avoids epsilon issues)
    w = w / w.min()

    bt = sub.betweenness(weights=w.tolist())  # pass list is most robust

    # Freeman normalization (undirected)
    # norm = (n - 1) * (n - 2) / 2
    # bt = [b / norm for b in bt]

    return {sub.vs[i]["orig_vid"]: bt[i] for i in range(n)}

def knn_from_similarity(S: csr_matrix, k: int) -> csr_matrix:
    S = S.tocsr()
    rows, cols, data = [], [], []

    for i in range(S.shape[0]):
        start, end = S.indptr[i], S.indptr[i + 1]
        row_data = S.data[start:end]
        row_cols = S.indices[start:end]

        if len(row_data) > k:
            idx = np.argpartition(row_data, -k)[-k:]
            rows.extend([i] * k)
            cols.extend(row_cols[idx])
            data.extend(row_data[idx])
        else:
            rows.extend([i] * len(row_data))
            cols.extend(row_cols)
            data.extend(row_data)

    return csr_matrix((data, (rows, cols)), shape=S.shape)

In [ ]:
msigdb_adjacency_matrix = load_npz(f"output/msigdb_adjacency_matrix.npz")

In [ ]:
msigdb_similarity_matrix_filtered = knn_from_similarity(msigdb_adjacency_matrix,400)
msigdb_similarity_matrix_filtered = msigdb_similarity_matrix_filtered.minimum(msigdb_similarity_matrix_filtered.T)

In [ ]:
msigdb_distance_matrix_filtered = msigdb_similarity_matrix_filtered.copy()
msigdb_distance_matrix_filtered.data = 1.0 / msigdb_distance_matrix_filtered.data

In [ ]:
msigdb_similarity_igraph = igraph_from_sparse(msigdb_similarity_matrix_filtered)
msigdb_distance_igraph = igraph_from_sparse(msigdb_distance_matrix_filtered)

In [ ]:
msigdb_similarity_nxgraph = nx.from_scipy_sparse_array(msigdb_similarity_matrix_filtered)
msigdb_distance_nxgraph = nx.from_scipy_sparse_array(msigdb_distance_matrix_filtered)

# Create plots

In [ ]:
def score_to_percentile(strength):
    return  rankdata(strength) / len(strength)

In [ ]:
def layer_comparison_bar_plot(
    ax,
    dgidb_scores,
    non_dgidb_scores,
    centrality,
    agg="mean"
):
    if agg == "mean":
        vals = [
            np.mean(dgidb_scores),
            np.mean(non_dgidb_scores)
        ]
    elif agg == "median":
        vals = [
            np.median(dgidb_scores),
            np.median(non_dgidb_scores)
        ]
    else:
        raise ValueError("agg must be 'mean' or 'median'")

    sems = [
        np.std(dgidb_scores, ddof=1) / np.sqrt(len(dgidb_scores)),
        np.std(non_dgidb_scores, ddof=1) / np.sqrt(len(non_dgidb_scores))
    ]

    ax.bar(
        ["DGIDB", "Non-DGIDB"],
        vals,
        yerr=sems,
        capsize=5
    )

    ax.set_ylabel(f"{agg} {centrality}")
    ax.set_title(f"{centrality}")


In [ ]:
def dgidb_centrality(centrality, cutoff = 400):
    if (centrality == "eigenvector"):
        score = msigdb_similarity_igraph.eigenvector_centrality(weights="weight")
    # elif (centrality == "closeness"):
    #     score = msigdb_distance_igraph.closeness(weights="weight")
    elif (centrality == "degree"):
        score = msigdb_similarity_igraph.strength(weights="weight")
    elif (centrality == "pagerank"):
        score = msigdb_similarity_igraph.pagerank(weights="weight")
    elif (centrality == "betweenness"):
        score = msigdb_distance_igraph.betweenness(weights="weight", cutoff = cutoff)
    

    
    return score
    

In [ ]:
def split_score(score,dgidb_index_in_msigdb):
    dgidb_scores = [score[i] for i in dgidb_index_in_msigdb]
    other_scores = [score[i] for i in range(len(score)) if i not in dgidb_index_in_msigdb]
    return dgidb_scores,other_scores

In [ ]:
dgidb_score_dict, other_score_dict = {},{}

In [ ]:
ec_scores = dgidb_centrality("eigenvector")

In [ ]:
strength_scores = dgidb_centrality("degree")

In [ ]:
pr_scores = dgidb_centrality("pagerank")

In [ ]:
bt_scores = dgidb_centrality("betweenness",400)

In [ ]:
scores = {}
scores["degree"] = strength_scores
scores["pagerank"] = pr_scores
scores["betweenness"] = bt_scores

In [ ]:
disease_list = ["BIPOLAR","BREASTCANCER","LEUKEMIA","SCHIZOPHRENIA"]
disease_list_common = ["Bipolar Disorder", "Breast Cancer", "Leukemia", "Schizophrenia"]
# disease_list = ["BIPOLAR"]
# disease_list_common = ["Bipolar Disorder"]

In [ ]:
# setting universal font sizes
font_size = 20
tick_font_size = 16
plt.rcParams.update({
    "font.size": font_size,          # base font size
    "axes.titlesize": font_size,
    "axes.labelsize": font_size,
    "xtick.labelsize": tick_font_size,
    "ytick.labelsize": tick_font_size,
    "legend.fontsize": font_size,
    "figure.titlesize": font_size,
    "legend.loc": 'best'
})


In [ ]:
centralities = ["degree", "pagerank", "betweenness"]
whitneyu_results = {}
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for k, cent in enumerate(centralities):
    ax = axes[k]
    ax.set_title(cent.capitalize() + " Centrality")
    
    bar1 = []
    bar2 = []
    sems1 = []
    sems2 = []
    
    for d in disease_list:
        # Load mapping from DGIDB to MSIGDB indices
        with open(f"output/{d}/dgidb_to_msigdb_indices_dict.json", "r") as f:
            dgidb_index_in_msigdb = json.load(f)
        # Convert index to intergers
        dgidb_index_in_msigdb = {int(i): j for i, j in dgidb_index_in_msigdb.items()}
        
        dgidb_scores, non_dgidb_scores = split_score(scores[cent], dgidb_index_in_msigdb)
        
        bar1.append(np.mean(dgidb_scores))
        bar2.append(np.mean(non_dgidb_scores))
        
        # one-sided Mann-Whitney U test (alternative='greater' means DGIDB > non-DGIDB)
        stat, p = mannwhitneyu(dgidb_scores, non_dgidb_scores, alternative="greater")

        auc = stat / (len(dgidb_scores) * len(non_dgidb_scores))
        whitneyu_results[(d, cent)] = {
            "p_value": p,
            "AUC": auc,
        }


        sems1.append(np.std(dgidb_scores, ddof=1) / np.sqrt(len(dgidb_scores)))
        sems2.append(np.std(non_dgidb_scores, ddof=1) / np.sqrt(len(non_dgidb_scores)))
    
    # Bar positions
    inner_spacing = 1.0
    group_gap = 7

    x = np.arange(len(bar1)) * inner_spacing
    x1 = x
    x2 = x + group_gap

    colors = plt.get_cmap("tab10")(range(len(disease_list)))

    # Plot DGIDB
    for i in range(len(disease_list)):
        ax.bar(x1[i], bar1[i], yerr=sems1[i], color=colors[i])

    # Plot non-DGIDB
    for i in range(len(disease_list)):
        ax.bar(x2[i], bar2[i], yerr=sems2[i], color=colors[i])

    # ---- FIX X AXIS: remove numbers, add two block labels ----
    center_left  = (x1[0] + x1[-1]) / 2
    center_right = (x2[0] + x2[-1]) / 2

    ax.set_xticks([center_left, center_right])
    ax.set_xticklabels(["DGIdb genes", "non-DGIdb genes"])

    # optional: remove tick marks too (cleaner)
    ax.tick_params(axis="x", length=0)

# Legend
legend_handles = [
    Patch(facecolor=colors[i], label=disease_list_common[i])
    for i in range(len(disease_list))
]

# create whitespace on the right (so legend is truly outside subplots)
fig.subplots_adjust(right=0.80)

# legend placed in the whitespace (NOT inside any subplot)
fig.legend(
    handles=legend_handles,
    loc="center left",
    bbox_to_anchor=(0.82, 0.8),  # outside axes, inside figure
    frameon=False
)

plt.savefig("../Graphs/dgidb_gene_centrality_comparison.pdf", bbox_inches="tight")

plt.show()



In [ ]:
whitneyu_results